# Molecular-graph EDA from MoleculeNet

Load a large molnet dataset and convert each molecule to a JSON-friendly
`{atoms, bonds}` graph via RDKit.

In [1]:
import deepchem as dc
import numpy as np
import pandas as pd
import json
from rdkit import Chem

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with transformers dependency. No module named 'transformers'
cannot import name 'HuggingFaceModel' from 'deepchem.models.torch_models' (/Users/reid/py/deepchemBenchmark/.venv/lib/python3.13/site-packages/deepchem/models/torch_models/__init__.py)
Skipped loading modules with pytorch-geome

## Load ZINC15 (250K slice)

`featurizer="Raw"` skips numeric featurization and stores the `rdkit.Chem.Mol`
object directly on `dataset.X`, which is exactly what we need. Swap `dataset_size`
to `"1M"` / `"10M"` / `"270M"` when you want more molecules — the first run of each
slice caches under `~/.deepchem/`.

In [2]:
tasks, datasets, transformers = dc.molnet.load_zinc15(
    featurizer="Raw",
    dataset_size="250K",
    reload=True,
)
train, valid, test = datasets
print("tasks:", tasks)
print("train / valid / test:", len(train), len(valid), len(test))
print("first X entry type:", type(train.X[0]).__name__)
print("first SMILES id:", train.ids[0])

tasks: ['mwt', 'logp', 'reactive']
train / valid / test: 200000 25000 25000
first X entry type: Mol
first SMILES id: ZINC001371026073


## SMILES → atom-centric dict

Uses RDKit atom / bond iterators. Bonds are stored as a child dict on each atom
so the graph is walkable from any node without a separate edge table. Each bond
appears twice — once under each endpoint — sharing the same `b{idx}` key so the
two views can be cross-referenced.

- atom: element, atomic number, formal charge, H count, radicals, chirality, aromatic flag, `bonds`
- atom.bonds[b{idx}]: neighbor atom key, bond order, stereo

In [ ]:
from rdkit.Chem import rdMolDescriptors, inchi

def mol_to_graph_dict(mol: Chem.Mol) -> dict:
    atoms = {}
    for a in mol.GetAtoms():
        atoms[f"a{a.GetIdx()}"] = {
            "element":             a.GetSymbol(),
            "atomic_number":       a.GetAtomicNum(),
            "formal_charge":       a.GetFormalCharge(),
            "n_hydrogens":         a.GetTotalNumHs(),
            "n_radical_electrons": a.GetNumRadicalElectrons(),
            "chirality":           a.GetChiralTag().name,
            "aromatic":            a.GetIsAromatic(),
            "bonds":               {},
        }

    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bkey  = f"b{b.GetIdx()}"
        props = {
            "order":  b.GetBondTypeAsDouble(),
            "stereo": b.GetStereo().name,
        }
        atoms[f"a{i}"]["bonds"][bkey] = {"atom": f"a{j}", **props}
        atoms[f"a{j}"]["bonds"][bkey] = {"atom": f"a{i}", **props}

    structure = {
        "nodes": list(atoms.keys()),
        "edges": {f"b{b.GetIdx()}": [f"a{b.GetBeginAtomIdx()}", f"a{b.GetEndAtomIdx()}"]
                  for b in mol.GetBonds()},
    }

    meta = {
        "canonical_smiles": Chem.MolToSmiles(mol),
        "inchi":            inchi.MolToInchi(mol),
        "inchi_key":        inchi.MolToInchiKey(mol),
        "formula":          rdMolDescriptors.CalcMolFormula(mol),
        "exact_mass":       rdMolDescriptors.CalcExactMolWt(mol),
        "net_charge":       Chem.GetFormalCharge(mol),
    }

    return {"atoms": atoms, "structure": structure, **meta}

def smiles_to_graph_dict(smi: str) -> dict | None:
    mol = Chem.MolFromSmiles(smi)
    return None if mol is None else mol_to_graph_dict(mol)

In [26]:
# Sanity-check on one molecule.
def record_for(dataset, idx: int) -> dict:
    return {
        "smiles":  str(dataset.ids[idx]),

        "targets": {task: float(v) for task, v in zip(tasks, dataset.y[idx])},

        **mol_to_graph_dict(dataset.X[idx]),
    }

sample_record = record_for(train, 0)

print(f"smiles: {sample_record['smiles']}")

print(f"atoms: {len(sample_record['atoms'])}")

smiles: ZINC001371026073
atoms: 24


In [27]:
sample_record

{'smiles': 'ZINC001371026073',
 'targets': {'mwt': 0.9223189221632002,
  'logp': -1.182089049790596,
  'reactive': -0.33830244114375835},
 'atoms': {'a0': {'element': 'C',
   'atomic_number': 6,
   'formal_charge': 0,
   'n_hydrogens': 3,
   'n_radical_electrons': 0,
   'chirality': 'CHI_UNSPECIFIED',
   'aromatic': False,
   'bonds': {'b0': {'atom': 'a3', 'order': 1.0, 'stereo': 'STEREONONE'}}},
  'a1': {'element': 'C',
   'atomic_number': 6,
   'formal_charge': 0,
   'n_hydrogens': 0,
   'n_radical_electrons': 0,
   'chirality': 'CHI_UNSPECIFIED',
   'aromatic': True,
   'bonds': {'b14': {'atom': 'a22', 'order': 1.0, 'stereo': 'STEREONONE'},
    'b15': {'atom': 'a4', 'order': 1.5, 'stereo': 'STEREONONE'},
    'b23': {'atom': 'a19', 'order': 1.5, 'stereo': 'STEREONONE'}}},
  'a2': {'element': 'C',
   'atomic_number': 6,
   'formal_charge': 0,
   'n_hydrogens': 1,
   'n_radical_electrons': 0,
   'chirality': 'CHI_UNSPECIFIED',
   'aromatic': True,
   'bonds': {'b20': {'atom': 'a8', 'or

In [28]:
from pathlib import Path

out = Path("zinc15_graphs.jsonl")
with out.open("w") as f:
    json.dump(sample_record, f, indent=2)

print("Saved to zinc15_graphs.jsonl")

Saved to zinc15_graphs.jsonl


## Bulk conversion + basic EDA

Convert the whole train split in one pass, then look at simple size distributions.
On the 250K slice this takes a few seconds; on 1M+ you'll want to stream to disk
(see the last cell) rather than keep everything in memory.

In [5]:
graphs = [mol_to_graph_dict(m) for m in train.X]

sizes = pd.DataFrame({
    "n_atoms": [len(g["atoms"]) for g in graphs],
    "n_bonds": [len(g["bonds"]) for g in graphs],
})
sizes.describe()

[18:42:13] WARNING: Charges were rearranged

[18:42:13] WARNING: Omitted undefined stereo

[18:42:13] WARNING: Charges were rearranged

[18:42:13] WARNING: Omitted undefined stereo

[18:42:13] WARNING: Omitted undefined stereo

[18:42:13] WARNING: Charges were rearranged

[18:42:13] WARNING: Charges were rearranged

[18:42:13] WARNING: Charges were rearranged

[18:42:13] WARNING: Omitted undefined stereo

[18:42:13] WARNING: Omitted undefined stereo

[18:42:13] WARNING: Charges were rearranged

[18:42:13] WARNING: Omitted undefined stereo

[18:42:13] WARNING: Omitted undefined stereo

[18:42:13] WARNING: Omitted undefined stereo

[18:42:13] WARNING: Charges were rearranged

[18:42:13] WARNING: Charges were rearranged

[18:42:13] WARNING: Omitted undefined stereo

[18:42:13] WARNING: Omitted undefined stereo

[18:42:13] WARNING: Omitted undefined stereo

[18:42:13] WARNING: Charges were rearranged; Omitted undefined stereo

[18:42:13] WARNING: Omitted undefined stereo

[18:42:14] WARNIN

,n_atoms,n_bonds
count,200000.000000,200000.000000
mean,21.969935,23.344235
std,2.094260,2.617169
min,10.000000,9.000000
25%,21.000000,22.000000
50%,22.000000,24.000000
75%,24.000000,25.000000
max,26.000000,31.000000


In [6]:
# Element frequency across the whole train split.
from collections import Counter
element_counts = Counter()
for g in graphs:
    element_counts.update(a["element"] for a in g["atoms"].values())

pd.Series(element_counts).sort_values(ascending=False).head(20)

C     3073101
N      695473
O      524881
F       48695
S       38644
Cl      11328
Br       1745
I          48
P          32
Si         27
B          13
dtype: int64

In [ ]:
# Bond-type distribution. Dedupe by bond key since each bond appears under both endpoints.
bond_counts = Counter()
for g in graphs:
    seen = {}
    for atom in g["atoms"].values():

        for bkey, bond in atom["bonds"].items():pd.Series(bond_counts).sort_index()

            seen.setdefault(bkey, bond["order"])    bond_counts.update(seen.values())

1.0    3092366
1.5    1203473
2.0     359544
3.0      13464
dtype: int64

In [ ]:
# cspell:ignore chirality CHI_UNSPECIFIED STEREONONE STEREOANY

# Summarize how often each nested field is recorded and meaningfully populated.

path_placeholders = {
    "atom.chirality": {"CHI_UNSPECIFIED"},
    "atom.bond.stereo": {"STEREONONE", "STEREOANY"},
}

def is_meaningful(path, v):
    if v is None:
        return False
    if isinstance(v, str):
        stripped = v.strip()
        if not stripped:
            return False
        return stripped not in path_placeholders.get(path, set())
    if isinstance(v, (list, tuple, set, dict)):
        return len(v) > 0

    na_flag = pd.isna(v)
    if isinstance(na_flag, (bool, np.bool_)):
        return not na_flag
    return True

def normalize_for_count(v):
    if isinstance(v, list):
        return tuple(v)
    if isinstance(v, dict):
        return tuple(sorted(v.items()))
    if isinstance(v, set):
        return tuple(sorted(v))
    return v

stats = {}  # path -> dict(scope, records, meaningful, all_values Counter, values Counter)
total_molecules = len(graphs)
total_atoms = 0
total_bonds = 0

def touch(path, scope, value):
    if path not in stats:
        stats[path] = {
            "scope": scope,
            "records": 0,
            "meaningful": 0,
            "all_values": Counter(),
            "values": Counter(),  # meaningful values only
        }
    s = stats[path]
    s["records"] += 1

    norm_value = normalize_for_count(value)
    s["all_values"][norm_value] += 1

    if is_meaningful(path, value):
        s["meaningful"] += 1
        s["values"][norm_value] += 1

for g in graphs:
    # molecule-level fields (everything at the root except the atoms container)
    for k, v in g.items():
        if k == "atoms":
            continue
        touch(f"meta.{k}", "molecule", v)

    # atom-level + nested bond-level fields
    atoms = g.get("atoms", {})
    total_atoms += len(atoms)
    seen_bonds = set()
    for atom in atoms.values():
        for k, v in atom.items():
            if k == "bonds":
                continue
            touch(f"atom.{k}", "atom", v)
        for bkey, bond in atom.get("bonds", {}).items():
            if bkey in seen_bonds:
                continue
            seen_bonds.add(bkey)
            for k, v in bond.items():
                touch(f"atom.bond.{k}", "bond", v)
    total_bonds += len(seen_bonds)

possible_by_scope = {
    "molecule": total_molecules,
    "atom": total_atoms,
    "bond": total_bonds,
}

rows = []
for path, s in stats.items():
    possible = possible_by_scope[s["scope"]]
    rows.append({
        "field": path,
        "scope": s["scope"],
        "possible_records": possible,
        "records_with_key": s["records"],
        "meaningful_records": s["meaningful"],
        "key_coverage": s["records"] / possible if possible else np.nan,
        "meaningful_coverage": s["meaningful"] / possible if possible else np.nan,
        "nunique_recorded_values": len(s["all_values"]),      # requested
        "nunique_meaningful_values": len(s["values"]),
        "top_recorded_values": s["all_values"].most_common(5),
        "top_meaningful_values": s["values"].most_common(5),
    })

field_stats = pd.DataFrame(rows).sort_values(
    ["scope", "meaningful_coverage", "nunique_recorded_values"],
    ascending=[True, True, True]
).reset_index(drop=True)

# Fields that never carry meaningful data
drop_never_meaningful = field_stats[field_stats["meaningful_records"] == 0]

# Optional: fields with only one meaningful value across the entire dataset
low_information = field_stats[
    (field_stats["meaningful_records"] > 0) &
    (field_stats["nunique_meaningful_values"] == 1)
]

print("Molecules:", total_molecules, "Atoms:", total_atoms, "Bonds:", total_bonds)
print("\nFields with no meaningful data:")
display(
    drop_never_meaningful[
        ["field", "scope", "possible_records", "records_with_key", "meaningful_records", "nunique_recorded_values"]
    ]
)

print("\nLow-information fields (single meaningful value):")
display(
    low_information[
        ["field", "scope", "meaningful_records", "nunique_recorded_values", "nunique_meaningful_values", "top_meaningful_values"]

    ]display(field_stats)

)# Full table for review


Molecules: 200000 Atoms: 4393987 Bonds: 4668847

Fields with no meaningful data:


,field,scope,possible_records,records_with_key,meaningful_records,nunique_recorded_values



Low-information fields (single meaningful value):


,field,scope,meaningful_records,nunique_recorded_values,nunique_meaningful_values,top_meaningful_values
1,atom.isotope,atom,4393987,1,1,"[(0, 4393987)]"
2,atom.atom_map_number,atom,4393987,1,1,"[(0, 4393987)]"


,field,scope,possible_records,records_with_key,meaningful_records,key_coverage,meaningful_coverage,nunique_recorded_values,nunique_meaningful_values,top_recorded_values,top_meaningful_values
0,atom.chirality,atom,4393987,4393987,285066,1.0,0.064876,3,2,"[(CHI_UNSPECIFIED, 4108921), (CHI_TETRAHEDRAL_...","[(CHI_TETRAHEDRAL_CW, 142701), (CHI_TETRAHEDRA..."
1,atom.isotope,atom,4393987,4393987,4393987,1.0,1.000000,1,1,"[(0, 4393987)]","[(0, 4393987)]"
2,atom.atom_map_number,atom,4393987,4393987,4393987,1.0,1.000000,1,1,"[(0, 4393987)]","[(0, 4393987)]"
3,atom.n_radical_electrons,atom,4393987,4393987,4393987,1.0,1.000000,2,2,"[(0, 4393979), (1, 8)]","[(0, 4393979), (1, 8)]"
4,atom.aromatic,atom,4393987,4393987,4393987,1.0,1.000000,2,2,"[(False, 3203959), (True, 1190028)]","[(False, 3203959), (True, 1190028)]"
5,atom.formal_charge,atom,4393987,4393987,4393987,1.0,1.000000,3,3,"[(0, 4388590), (1, 2708), (-1, 2689)]","[(0, 4388590), (1, 2708), (-1, 2689)]"
6,atom.n_hydrogens,atom,4393987,4393987,4393987,1.0,1.000000,4,4,"[(0, 1789797), (1, 1139918), (2, 1080963), (3,...","[(0, 1789797), (1, 1139918), (2, 1080963), (3,..."
7,atom.element,atom,4393987,4393987,4393987,1.0,1.000000,11,11,"[(C, 3073101), (N, 695473), (O, 524881), (F, 4...","[(C, 3073101), (N, 695473), (O, 524881), (F, 4..."
8,atom.atomic_number,atom,4393987,4393987,4393987,1.0,1.000000,11,11,"[(6, 3073101), (7, 695473), (8, 524881), (9, 4...","[(6, 3073101), (7, 695473), (8, 524881), (9, 4..."
9,bond.stereo,bond,4668847,4668847,4882,1.0,0.001046,3,2,"[(STEREONONE, 4663965), (STEREOE, 2580), (STER...","[(STEREOE, 2580), (STEREOZ, 2302)]"


## Streaming to disk for very large datasets

For anything above ~1 M molecules the in-memory list becomes wasteful. Write
one JSON object per line (JSONL) so the file can be read lazily by any downstream
tool (pandas `read_json(..., lines=True)`, `duckdb`, `pyarrow`, etc.).

In [ ]:
from pathlib import Path

out = Path("zinc15_graphs.jsonl")
with out.open("w") as f:
    for i in range(len(train)):
        f.write(json.dumps(record_for(train, i)) + "\n")
print(f"wrote {out}  ({out.stat().st_size / 1e6:.1f} MB)")